<a href="https://colab.research.google.com/github/santachan/santachan.github.io/blob/main/AI_Lunal_Landing_Moon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install gymnasium

In [ ]:
import gymnasium as gym

In [ ]:
pip install swig

  Using cached swig-4.4.1-py3-none-manylinux_2_12_x86_64.manylinux2010_x86_64.whl.metadata (3.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 79.1 MB/s eta 0:00:00


In [ ]:
pip install "gymnasium[box2d]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 71.0 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.autograd as autograd
from torch.autograd import Variable
from collections import deque, namedtuple

In [ ]:
env = gym.make("LunarLander-v3", continuous=False, gravity=-10.0,
               enable_wind=False, wind_power=15.0, turbulence_power=1.5)

In [ ]:
state = env.observation_space.shape
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
print(state)
print(state_size)
print(action_size)

(8,)
8
4


In [ ]:
learning_rate = 5e-4
minibatch = 150
gamma = 0.99
replay_buffer_size = 100000
interpolation_parameter = 1e-3
number_episodes = 5000
max_time_steps = 1000
epsilon_starting_value = 1.0
epsilon_ending_value = 0.01
epsilon_decay_value = 0.995
scores_100_episodes = deque(maxlen=100)

In [ ]:
class ANN(nn.Module):
  def __init__(self, state_size, action_size, seed=42):
    super(ANN, self).__init__()
    self.seed = torch.manual_seed(seed)
    self.fc1 = nn.Linear(state_size, 64)
    self.fc2 = nn.Linear(64, 64)
    self.fc3 = nn.Linear(64, action_size)

  def forward(self, state):
    x = self.fc1(state)
    x = F.relu(x)
    x = self.fc2(x)
    x = F.relu(x)
    return self.fc3(x)



In [ ]:
class ReplayMemory(object):

  def __init__(self, capacity):
    self.capacity = capacity
    self.memory = []

  # push event to memory
  def push(self, event):
      self.memory.append(event)
      # 만약 capacity보다 메모리 길이가 길면 오래된 데이터를 지움
      if len(self.memory) > self.capacity:
          del self.memory[0]

  #에이전트가 배우기 위해 선택할 랜덤 experienc를 선택하는 역할
  def sample(self, batch_size):
    experiences = random.sample(self.memory, batch_size)
    states = torch.from_numpy(np.vstack([e[0] for e in experiences if e is not None])).float()
    actions = torch.from_numpy(np.vstack([e[1] for e in experiences if e is not None])).long()
    rewards = torch.from_numpy(np.vstack([e[2] for e in experiences if e is not None])).float()
    next_states = torch.from_numpy(np.vstack([e[3] for e in experiences if e is not None])).float()
    dones = torch.from_numpy(np.vstack([e[4] for e in experiences if e is not None]).astype(np.uint8)).float()

    return states, actions, rewards, next_states, dones

In [ ]:
class Agent():

  def __init__(self, state_size, action_size):
    self.state_size = state_size
    self.action_size = action_size
    self.local_qnetwork = ANN(state_size, action_size)
    self.target_qnetwork = ANN(state_size, action_size)
    self.optimizer = optim.Adam(self.local_qnetwork.parameters(), lr = learning_rate)
    self.memory = ReplayMemory(replay_buffer_size)
    self.t_step = 0

  def step(self, state, action, reward, next_state, done):
    self.memory.push((state, action, reward, next_state, done))
    self.t_step = (self.t_step + 1) % 4
    if self.t_step == 0:
      if len(self.memory.memory) > minibatch:
        experience = self.memory.sample(minibatch)
        self.learn(experience, gamma)

  def get_action(self, state, epsilon):
    # unsqueeze(0) = tensor 앞에 batch 차원을 추가하는 함수
    state = torch.from_numpy(state).float().unsqueeze(0)
    self.local_qnetwork.eval()
    with torch.no_grad():
      action_values = self.local_qnetwork(state)
    self.local_qnetwork.train()
    #여기가 Exploitation 부분
    if random.random() > epsilon:
      return np.argmax(action_values.cpu().data.numpy())
    #여기가 Exploration 부분
    else:
      return random.choice(np.arange(self.action_size)) #앞에서 방향 앞뒤좌우를 랜덤으로 선택

  def learn(self, experiences, gamma):
    states, actions, rewards, next_states, dones = experiences
    next_q_targets = self.target_qnetwork(next_states).detach().max(1)[0].unsqueeze(1)
    q_targets = rewards + (gamma * next_q_targets * (1 - dones))
    q_expected = self.local_qnetwork(states).gather(1, actions)
    loss = F.mse_loss(q_expected, q_targets)
    self.optimizer.zero_grad()
    loss.backward()
    self.optimizer.step()
    self.soft_update(self.local_qnetwork, self.target_qnetwork, interpolation_parameter)

  def soft_update(self, local_qnetwork, target_qnetwork, interpolation_parameter):
    for target_param, local_param in zip(target_qnetwork.parameters(), local_qnetwork.parameters()):
      target_param.data.copy_(interpolation_parameter * local_param.data + (1.0 - interpolation_parameter) * target_param.data)


In [ ]:
agent = Agent(state_size, action_size)

In [ ]:
epsilon = epsilon_starting_value
for episode in range(0,number_episodes):
  state, _ = env.reset()
  score = 0
  for st in range(0, max_time_steps):
    action = agent.get_action(state, epsilon)
    next_state, reward, done, _, _ = env.step(action)
    agent.step(state, action, reward, next_state, done)
    state = next_state
    score += reward
    if done:
      break

  scores_100_episodes.append(score)
  epsilon = max(epsilon_ending_value, epsilon*epsilon_decay_value)

  if episode % 10 == 0:
    print('Episode{} Avg Score{:2f}'.format(episode, np.mean(scores_100_episodes)))
  if np.mean(scores_100_episodes) >= 200.0:
    print('Environment solved in {:d} episodes!\tAverage Score: {:2f}'.format(episode, np.mean(scores_100_episodes)))
    break

Episode0 Avg Score-12.841116
Episode10 Avg Score-10.482540
Episode20 Avg Score-12.312167
Episode30 Avg Score-23.352104
Episode40 Avg Score-43.115887
Episode50 Avg Score-56.716030
Episode60 Avg Score-67.562950
Episode70 Avg Score-76.861258
Episode80 Avg Score-86.225713
Episode90 Avg Score-93.846808
Episode100 Avg Score-98.259493
Episode110 Avg Score-89.403013
Episode120 Avg Score-78.379755
Episode130 Avg Score-73.353271
Episode140 Avg Score-59.126917
Episode150 Avg Score-58.315013
Episode160 Avg Score-50.870592
Episode170 Avg Score-48.679740
Episode180 Avg Score-53.207274
Episode190 Avg Score-52.734969
Episode200 Avg Score-57.571086
Episode210 Avg Score-56.989112
Episode220 Avg Score-55.190608
Episode230 Avg Score-53.631497
Episode240 Avg Score-53.184309
Episode250 Avg Score-40.393508
Episode260 Avg Score-36.340599
Episode270 Avg Score-31.436218
Episode280 Avg Score-11.853422
Episode290 Avg Score2.828859
Episode300 Avg Score22.619598
Episode310 Avg Score38.624563
Episode320 Avg Score53.

In [ ]:
import glob
import io
import base64
import imageio
import gymnasium as gym
from IPython.display import HTML, display


def record_agent_video(agent, env_name, output_filename="video.mp4", fps=30):
    """Records a video of the agent interacting with the environment."""

    env = gym.make(env_name, render_mode="rgb_array")
    state, _ = env.reset()

    done = False
    frames = []

    while not done:
        frames.append(env.render())  # Capture frame

        action = agent.get_action(state, 0.0)  # Get action from agent

        state, reward, done, _, _ = env.step(action.item())

    env.close()

    imageio.mimsave(output_filename, frames, fps=fps)  # Save video


def display_video(filename="video.mp4"):

    try:
        with open(filename, "rb") as video_file:
            encoded_video = base64.b64encode(video_file.read()).decode("ascii")

        display(HTML(f"""
        <video alt="Agent Playing" autoplay loop controls style="height: 400px;">
            <source src="data:video/mp4;base64,{encoded_video}" type="video/mp4" />
        </video>
        """))

    except FileNotFoundError:
        print("Error: Video file not found!")


# Example usage
record_agent_video(agent, "LunarLander-v3")
display_video()